In [ ]:
# ============================================================
# NOTEBOOK CELL 1
# Generate intermediate files only:
#   - climatology cache
#   - BWCN reference cache
#   - Hindcast cache
#   - metrics txt
# No plotting here.
# ============================================================

import os
import re
import glob
import gc
import datetime

import numpy as np
import xarray as xr
from tqdm import tqdm

# ============================================================
# CONFIG
# ============================================================

DATA_ROOT = "/mnt/soclim0/public_data/weiji"

HINDCAST_ROOT = os.path.join(DATA_ROOT, "Hindcast")
BWCN_ROOT = os.path.join(DATA_ROOT, "BWCN")
B2000_ROOT = os.path.join(DATA_ROOT, "B2000WCN001002")

OUT_ROOT = "/home/weiji/restart_exam/code/20260415egu/plots/hindcast/O3"

USE_CACHE = True
FORCE_REBUILD_O3 = False
FORCE_REBUILD_T = False
FORCE_REBUILD_U = False
FORCE_REBUILD_U50 = False
FORCE_REBUILD_BWCN = False
FORCE_REBUILD_CLIM = False

OPEN_ENGINE = "netcdf4"

XLIM_START = 0
XLIM_END = 150



# O3 only
O3_PTOP = 30.0
O3_PBOT = 70.0
O3_TAG = "30_70hPa"

CACHE_ROOT = os.path.join(OUT_ROOT, "cache")
FIG_ROOT = os.path.join(OUT_ROOT, "figures")

for sub in [
    CACHE_ROOT,
    FIG_ROOT,
    os.path.join(CACHE_ROOT, "O3"),
    os.path.join(CACHE_ROOT, "Tmin50"),
    os.path.join(CACHE_ROOT, "U60N10"),
    os.path.join(CACHE_ROOT, "U60N50"),
    os.path.join(FIG_ROOT, "O3"),
    os.path.join(FIG_ROOT, "Tmin50"),
    os.path.join(FIG_ROOT, "U60N10"),
]:
    os.makedirs(sub, exist_ok=True)

# ============================================================
# FIGURE SPECS
# ============================================================

FIGURE_SPECS = [
    {
        "year": "0008",
        "prefix": "O3_evolution_fig1",
        "experiments": [
            ("0008-01", "0008Jan_small_pert", 0,  "forestgreen"),
            ("0008-02", "0008Feb_small_pert", 31, "royalblue"),
            ("0008-03", "0008Mar_small_pert", 59, "hotpink"),
        ],
    },
    {
        "year": "0008",
        "prefix": "O3_evolution_fig2",
        "experiments": [
            ("0008-02_v2", "0008Feb_large_pert", 31, "forestgreen"),
            ("0008-03_v2", "0008Mar_large_pert", 59, "royalblue"),
            ("0008-04_v2", "0008Apr_large_pert", 90, "hotpink"),
        ],
    },
    {
        "year": "0008",
        "prefix": "O3_evolution_fig3",
        "experiments": [
            ("0008-02",    "0008Feb_small_pert", 31, "forestgreen"),
            ("0008-02_v2", "0008Feb_large_pert", 31, "royalblue"),
            ("0008-02_v3", "0008Feb_moist_pert", 31, "hotpink"),
        ],
    },
    {
        "year": "0008",
        "prefix": "O3_evolution_fig4",
        "experiments": [
            ("0008-03",    "0008Mar_small_pert", 59, "forestgreen"),
            ("0008-03_v2", "0008Mar_large_pert", 59, "royalblue"),
            ("0008-03_v3", "0008Mar_moist_pert", 59, "hotpink"),
        ],
    },
    {
        "year": "0003",
        "prefix": "O3_evolution_fig5",
        "experiments": [
            ("0003-02", "0003Feb_small_pert", 31, "forestgreen"),
            ("0003-03", "0003Mar_small_pert", 59, "royalblue"),
        ],
    },
    {
        "year": "0013",
        "prefix": "O3_evolution_fig6",
        "experiments": [
            ("0013-02", "0013Feb_small_pert", 31, "forestgreen"),
            ("0013-03", "0013Mar_small_pert", 59, "royalblue"),
        ],
    },
    {
        "year": "0014",
        "prefix": "O3_evolution_fig7",
        "experiments": [
            ("0014-02", "0014Feb_small_pert", 31, "forestgreen"),
            ("0014-03", "0014Mar_small_pert", 59, "royalblue"),
        ],
    },
    {
        "year": "0019",
        "prefix": "O3_evolution_fig8",
        "experiments": [
            ("0019-02", "0019Feb_small_pert", 31, "forestgreen"),
            ("0019-03", "0019Mar_small_pert", 59, "royalblue"),
        ],
    },
    {
        "year": "0008",
        "prefix": "O3_evolution_fig9",
        "experiments": [
            ("0008-02_NOCOUPL", "0008Feb_nocouple", 31, "forestgreen"),
            ("0008-03_NOCOUPL", "0008Mar_nocouple", 59, "royalblue"),
        ],
    },
    {
        "year": "0013",
        "prefix": "O3_evolution_fig10",
        "experiments": [
            ("0013-02_NOCOUPL", "0013Feb_nocouple", 31, "forestgreen"),
            ("0013-03_NOCOUPL", "0013Mar_nocouple", 59, "royalblue"),
        ],
    },
    {
        "year": "0014",
        "prefix": "O3_evolution_fig11",
        "experiments": [
            ("0014-02_NOCOUPL", "0014Feb_nocouple", 31, "forestgreen"),
            ("0014-03_NOCOUPL", "0014Mar_nocouple", 59, "royalblue"),
        ],
    },
    {
        "year": "0019",
        "prefix": "O3_evolution_fig12",
        "experiments": [
            ("0019-02_NOCOUPL", "0019Feb_nocouple", 31, "forestgreen"),
            ("0019-03_NOCOUPL", "0019Mar_nocouple", 59, "royalblue"),
        ],
    },
    {
        "year": "0008",
        "prefix": "O3_evolution_fig13",
        "experiments": [
            ("0008-02",         "0008Feb_small_pert", 31, "forestgreen"),
            ("0008-02_NOCOUPL", "0008Feb_nocouple",   31, "royalblue"),
        ],
    },
    {
        "year": "0008",
        "prefix": "O3_evolution_fig14",
        "experiments": [
            ("0008-03",         "0008Mar_small_pert", 59, "forestgreen"),
            ("0008-03_NOCOUPL", "0008Mar_nocouple",   59, "royalblue"),
        ],
    },
    {
        "year": "0013",
        "prefix": "O3_evolution_fig15",
        "experiments": [
            ("0013-02",         "0013Feb_small_pert", 31, "forestgreen"),
            ("0013-02_NOCOUPL", "0013Feb_nocouple",   31, "royalblue"),
        ],
    },
    {
        "year": "0013",
        "prefix": "O3_evolution_fig16",
        "experiments": [
            ("0013-03",         "0013Mar_small_pert", 59, "forestgreen"),
            ("0013-03_NOCOUPL", "0013Mar_nocouple",   59, "royalblue"),
        ],
    },
    {
        "year": "0014",
        "prefix": "O3_evolution_fig17",
        "experiments": [
            ("0014-02",         "0014Feb_small_pert", 31, "forestgreen"),
            ("0014-02_NOCOUPL", "0014Feb_nocouple",   31, "royalblue"),
        ],
    },
    {
        "year": "0014",
        "prefix": "O3_evolution_fig18",
        "experiments": [
            ("0014-03",         "0014Mar_small_pert", 59, "forestgreen"),
            ("0014-03_NOCOUPL", "0014Mar_nocouple",   59, "royalblue"),
        ],
    },
    {
        "year": "0019",
        "prefix": "O3_evolution_fig19",
        "experiments": [
            ("0019-02",         "0019Feb_small_pert", 31, "forestgreen"),
            ("0019-02_NOCOUPL", "0019Feb_nocouple",   31, "royalblue"),
        ],
    },
    {
        "year": "0019",
        "prefix": "O3_evolution_fig20",
        "experiments": [
            ("0019-03",         "0019Mar_small_pert", 59, "forestgreen"),
            ("0019-03_NOCOUPL", "0019Mar_nocouple",   59, "royalblue"),
        ],
    },
]

# ============================================================
# HELPERS
# ============================================================

def parse_member_id(path):
    base = os.path.basename(path)
    m = re.search(r"\.(\d{3})\.(?:[^\.]+\.)?cam\.h3", base)
    if m:
        return int(m.group(1))
    return None

def parse_b2000_file_year(path):
    base = os.path.basename(path)
    m = re.search(r"\.cam\.h3\.(\d{4})\.", base)
    if not m:
        raise ValueError(f"Cannot parse file year from: {base}")
    return int(m.group(1))

def get_lat_weights(ds_or_da, lat_slice):
    if "gw" in ds_or_da:
        return ds_or_da["gw"].sel(lat=lat_slice)
    lat = ds_or_da["lat"].sel(lat=lat_slice)
    return xr.DataArray(np.cos(np.deg2rad(lat)), dims=["lat"], coords={"lat": lat})

def find_bwcn_var_files_for_year(year_str, varname):
    root = os.path.join(BWCN_ROOT, varname)
    patterns = [
        os.path.join(root, f"BWCN.cam.h3.{year_str}.{varname}.nc"),
        os.path.join(root, f"BWCN.sample.cam.h3.{year_str}.{varname}.nc"),
        os.path.join(root, f"*{year_str}*.{varname}.nc"),
    ]
    for pat in patterns:
        files = sorted(glob.glob(pat))
        if files:
            return files
    raise FileNotFoundError(
        f"No BWCN {varname} files found for year {year_str} under {root}."
    )

def get_hindcast_files(exp_name, varname):
    root = os.path.join(HINDCAST_ROOT, exp_name, varname)
    files = sorted(glob.glob(os.path.join(root, f"*.{varname}.nc")))
    if not files:
        raise FileNotFoundError(f"No {varname} files found under {root}")
    return files

def get_b2000_files(varname):
    root = os.path.join(B2000_ROOT, varname)
    files = sorted(glob.glob(os.path.join(root, f"B2000WCN.sample.cam.h3.*.{varname}.nc")))
    if not files:
        raise FileNotFoundError(f"No B2000WCN {varname} files found under {root}")
    return files

def open_cached_dataarray(path):
    da = xr.open_dataarray(path)
    da.load()
    return da

def trim_dataset_for_diag(ds, diag_name):
    if diag_name == "O3":
        keep = {"O3", "PS", "P0", "hyai", "hybi", "gw", "time", "lat", "lon", "lev", "ilev"}
    elif diag_name == "Tmin50":
        keep = {"T", "PS", "P0", "hyam", "hybm", "gw", "time", "lat", "lon", "lev"}
    elif diag_name in ("U60N10", "U60N50"):
        keep = {"U", "PS", "P0", "hyam", "hybm", "gw", "time", "lat", "lon", "lev"}
    else:
        return ds

    drop_vars = [v for v in ds.variables if v not in keep]
    if drop_vars:
        ds = ds.drop_vars(drop_vars)
    return ds

# ============================================================
# METRICS HELPERS
# ============================================================

FW_LAT, FW_PLEV_HPA, FW_THRESHOLD, FW_MAX_CONSEC_WESTERLY = 60.0, 50.0, 7.0, 10

_base_date = datetime.datetime(2001, 1, 1)
MONTH_DAY_365 = [
    ((_base_date + datetime.timedelta(days=i)).month,
     (_base_date + datetime.timedelta(days=i)).day)
    for i in range(365)
]

def find_fw_for_one_year(vals_daily):
    vals_jj = np.asarray(vals_daily[:181], dtype=np.float64)
    for day_idx in range(len(vals_jj)):
        if not np.isfinite(vals_jj[day_idx]):
            continue
        if vals_jj[day_idx] < FW_THRESHOLD:
            has_long_westerly = False
            for i in range(day_idx + 1, max(day_idx + 1, len(vals_jj) - FW_MAX_CONSEC_WESTERLY + 1)):
                seg = vals_jj[i:i + FW_MAX_CONSEC_WESTERLY]
                if len(seg) < FW_MAX_CONSEC_WESTERLY:
                    break
                if np.all(seg > FW_THRESHOLD):
                    has_long_westerly = True
                    break
            if not has_long_westerly:
                doy = day_idx + 1
                return doy, MONTH_DAY_365[doy - 1][0], MONTH_DAY_365[doy - 1][1]
    return np.nan, np.nan, np.nan

def get_o3_ma_min_for_member(o3_da_1d, apply_o3_5d=True):
    if apply_o3_5d:
        o3_da_1d = o3_da_1d.rolling(time=5, center=True, min_periods=5).mean()

    mask = o3_da_1d.time.dt.month.isin([3, 4])
    seg = o3_da_1d.where(mask, drop=True)

    vals = seg.values
    valid = np.isfinite(vals)
    if not np.any(valid):
        return np.nan, np.nan, np.nan

    idx = np.nanargmin(vals)
    min_val = float(vals[idx])
    min_month = int(seg.time.dt.month.values[idx])
    min_day = int(seg.time.dt.day.values[idx])

    return min_val, min_month, min_day

# ============================================================
# HYBRID CORE
# ============================================================

def calc_parc_o3_hybrid(ds_sub, p_top_hpa, p_bot_hpa, verbose=False):
    g = 9.80665
    M_air = 28.964 / 1000.0
    Na = 6.02214e23
    DU = 2.687e20
    factor = Na / (g * M_air * DU)

    P0 = ds_sub["P0"].squeeze(drop=True)
    PS = ds_sub["PS"]

    P_interface = ds_sub["hyai"] * P0 + ds_sub["hybi"] * PS
    p_i = P_interface.isel(ilev=slice(0, -1)).rename({"ilev": "lev"})
    p_ip1 = P_interface.isel(ilev=slice(1, None)).rename({"ilev": "lev"})

    if "lev" in ds_sub.coords:
        lev_vals = ds_sub["lev"].values
        p_i = p_i.assign_coords(lev=lev_vals)
        p_ip1 = p_ip1.assign_coords(lev=lev_vals)

    p_layer_top = xr.where(p_i < p_ip1, p_i, p_ip1)
    p_layer_bot = xr.where(p_i > p_ip1, p_i, p_ip1)

    pT = p_top_hpa * 100.0
    pB = p_bot_hpa * 100.0

    upper = xr.where(p_layer_top > pT, p_layer_top, pT)
    lower = xr.where(p_layer_bot < pB, p_layer_bot, pB)
    overlap = xr.where(lower > upper, lower - upper, 0.0)

    if "lev" in ds_sub.coords:
        overlap = overlap.assign_coords(lev=ds_sub["lev"].values)

    O3_col = (ds_sub["O3"] * overlap * factor).sum(dim="lev")

    if verbose:
        _ = float((overlap.sum("lev") / 100.0).mean().load())

    return O3_col

def _interp_profile_logp(var_prof, p_prof, p_target_pa):
    mask = np.isfinite(var_prof) & np.isfinite(p_prof) & (p_prof > 0)
    if mask.sum() < 2:
        return np.nan

    x = np.log(p_prof[mask])
    y = var_prof[mask]

    order = np.argsort(x)
    x = x[order]
    y = y[order]

    xt = np.log(p_target_pa)
    if xt < x[0] or xt > x[-1]:
        return np.nan

    return np.interp(xt, x, y)

def interp_hybrid_to_pressure(ds, varname, p_target_hpa):
    P0 = ds["P0"].squeeze(drop=True)
    p_mid = ds["hyam"] * P0 + ds["hybm"] * ds["PS"]

    out = xr.apply_ufunc(
        _interp_profile_logp,
        ds[varname],
        p_mid,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[[]],
        vectorize=True,
        kwargs={"p_target_pa": p_target_hpa * 100.0},
        output_dtypes=[float],
    )
    return out

# ============================================================
# DIAGNOSTIC FUNCTIONS
# ============================================================

def compute_o3_pc_ts(ds):
    lat_slice = slice(60, 90)
    ds_sub = ds.sel(lat=lat_slice).load()
    O3_pc_3d = calc_parc_o3_hybrid(ds_sub, O3_PTOP, O3_PBOT)
    O3_zm = O3_pc_3d.mean(dim="lon")
    weights = get_lat_weights(ds_sub, lat_slice)
    O3_ts = O3_zm.weighted(weights).mean(dim="lat")
    O3_ts.name = "O3_pc_30_70hPa_60_90N"
    return O3_ts

def compute_polar_min_T50_ts(ds):
    ds_polar = ds.sel(lat=slice(60, 90)).load()
    T50 = interp_hybrid_to_pressure(ds_polar, "T", 50.0)
    T50_zm = T50.mean(dim="lon")
    Tmin50 = T50_zm.min(dim="lat")
    Tmin50.name = "Tmin50_60_90N"
    return Tmin50

def compute_U60N10_ts(ds):
    ds_60N = ds.sel(lat=slice(55, 65)).load()
    U10 = interp_hybrid_to_pressure(ds_60N, "U", 10.0)
    U10_zm = U10.mean(dim="lon")
    U60N10 = U10_zm.interp(lat=60.0)
    U60N10.name = "U60N_10hPa"
    return U60N10

def compute_U60N50_ts(ds):
    ds_60N = ds.sel(lat=slice(55, 65)).load()
    U50 = interp_hybrid_to_pressure(ds_60N, "U", 50.0)
    U50_zm = U50.mean(dim="lon")
    U60N50 = U50_zm.interp(lat=60.0)
    U60N50.name = "U60N_50hPa"
    return U60N50

# ============================================================
# CACHE BUILDERS
# ============================================================

def load_hindcast_diag(exp_name, varname, diag_name, compute_func,
                       use_cache=True, force_rebuild=False):
    cache_dir = os.path.join(CACHE_ROOT, diag_name)
    out_nc = os.path.join(cache_dir, f"{diag_name}_Hindcast_{exp_name}.nc")

    if use_cache and (not force_rebuild) and os.path.exists(out_nc):
        return open_cached_dataarray(out_nc)

    files = get_hindcast_files(exp_name, varname)
    member_pairs = sorted((parse_member_id(f), f) for f in files)

    ts_list = []

    for member_id, f in tqdm(member_pairs, desc=f"    {diag_name} {exp_name}", leave=False):
        with xr.open_dataset(f, engine=OPEN_ENGINE, cache=False) as ds:
            ds = trim_dataset_for_diag(ds, diag_name)
            ts_mem = compute_func(ds)
            ts_mem = ts_mem.load()
            ts_mem = ts_mem.expand_dims(member=[member_id])
            ts_list.append(ts_mem)
        gc.collect()

    ts = xr.concat(ts_list, dim="member").sortby("member")
    ts.to_netcdf(out_nc)
    return ts

def load_bwcn_reference_diag(year_str, varname, diag_name, compute_func,
                             use_cache=True, force_rebuild=False):
    cache_dir = os.path.join(CACHE_ROOT, diag_name)
    out_nc = os.path.join(cache_dir, f"{diag_name}_BWCN_{year_str}.nc")

    if use_cache and (not force_rebuild) and os.path.exists(out_nc):
        return open_cached_dataarray(out_nc)

    files = find_bwcn_var_files_for_year(year_str, varname)

    ts_parts = []
    for f in files:
        with xr.open_dataset(f, engine=OPEN_ENGINE, cache=False) as ds:
            ds = trim_dataset_for_diag(ds, diag_name)
            ts_part = compute_func(ds)
            ts_part = ts_part.load()
            ts_parts.append(ts_part)
        gc.collect()

    if len(ts_parts) == 1:
        ts = ts_parts[0]
    else:
        ts = xr.concat(ts_parts, dim="time").sortby("time")

    ts.to_netcdf(out_nc)
    return ts

def build_b2000_climatology_diag(varname, diag_name, compute_func,
                                 use_cache=True, force_rebuild=False):
    cache_dir = os.path.join(CACHE_ROOT, diag_name)
    ts_nc = os.path.join(cache_dir, f"{diag_name}_B2000WCN_ts.nc")
    clim_nc = os.path.join(cache_dir, f"{diag_name}_B2000WCN_climatology.nc")

    if use_cache and (not force_rebuild) and os.path.exists(ts_nc) and os.path.exists(clim_nc):
        ts = open_cached_dataarray(ts_nc)
        clim = open_cached_dataarray(clim_nc)
        return ts, clim

    print(f"      -> Building {diag_name} climatology year-by-year (Safe RAM mode)...")
    all_files = get_b2000_files(varname)

    valid_files = []
    for f in all_files:
        yr = parse_b2000_file_year(f)
        if (1 <= yr <= 103) or (105 <= yr <= 210):
            valid_files.append((yr, f))
    valid_files.sort(key=lambda x: x[0])

    ts_list = []

    for yr, f in tqdm(valid_files, desc=f"      Processing B2000 {diag_name}"):
        with xr.open_dataset(f, engine=OPEN_ENGINE, cache=False) as ds:
            ds = trim_dataset_for_diag(ds, diag_name)

            if yr >= 105:
                offset = 103
                new_times = [t.replace(year=t.year + offset) for t in ds.time.values]
                ds = ds.assign_coords(time=new_times)

            ts_yr = compute_func(ds)
            ts_yr = ts_yr.load()
            ts_list.append(ts_yr)

        gc.collect()

    ts = xr.concat(ts_list, dim="time").sortby("time")
    ts.to_netcdf(ts_nc)

    clim = ts.groupby("time.dayofyear").mean("time")
    clim.to_netcdf(clim_nc)

    return ts, clim

def build_diag_cache_suite(varname, diag_name, compute_func, force_rebuild_flag):
    print(f"\n================ {diag_name} CACHE SUITE =================\n")

    print(f"[{diag_name}] Step 1/3: Build climatology cache...")
    build_b2000_climatology_diag(
        varname=varname,
        diag_name=diag_name,
        compute_func=compute_func,
        use_cache=USE_CACHE,
        force_rebuild=FORCE_REBUILD_CLIM or force_rebuild_flag,
    )

    years_needed = sorted(set(spec["year"] for spec in FIGURE_SPECS))
    all_exp_names = sorted(set(exp_name for spec in FIGURE_SPECS for exp_name, _, _, _ in spec["experiments"]))

    print(f"[{diag_name}] Step 2/3: Build BWCN reference cache...")
    for year in tqdm(years_needed, desc=f"Reference {diag_name}"):
        load_bwcn_reference_diag(
            year_str=year,
            varname=varname,
            diag_name=diag_name,
            compute_func=compute_func,
            use_cache=USE_CACHE,
            force_rebuild=FORCE_REBUILD_BWCN or force_rebuild_flag,
        )
        gc.collect()

    print(f"[{diag_name}] Step 3/3: Build Hindcast cache...")
    for exp_name in tqdm(all_exp_names, desc=f"Hindcast {diag_name}"):
        load_hindcast_diag(
            exp_name=exp_name,
            varname=varname,
            diag_name=diag_name,
            compute_func=compute_func,
            use_cache=USE_CACHE,
            force_rebuild=force_rebuild_flag,
        )
        gc.collect()

# ============================================================
# METRICS EXPORT
# ============================================================

def process_metrics_for_exp(exp_name):
    try:
        o3_data = load_hindcast_diag(
            exp_name, "O3", "O3",
            compute_o3_pc_ts,
            use_cache=True,
            force_rebuild=False
        )
        u50_data = load_hindcast_diag(
            exp_name, "U", "U60N50",
            compute_U60N50_ts,
            use_cache=True,
            force_rebuild=False
        )

        out_file = os.path.join(HINDCAST_ROOT, exp_name, f"{exp_name}_metrics.txt")

        with open(out_file, "w") as f:
            f.write("Member\tFWD_DOY\tFWD_Date\tO3_MA_Min_Val\tO3_Min_Date\n")

            for m in o3_data.member.values:
                o3_m = o3_data.sel(member=m)
                u50_m = u50_data.sel(member=m)

                fwd_doy, fw_month, fw_day = find_fw_for_one_year(u50_m.values)
                fwd_date_str = f"{int(fw_month):02d}-{int(fw_day):02d}" if not np.isnan(fw_month) else "NaN"

                o3_val, o3_month, o3_day = get_o3_ma_min_for_member(o3_m)
                o3_date_str = f"{int(o3_month):02d}-{int(o3_day):02d}" if not np.isnan(o3_month) else "NaN"

                f.write(f"{m:03d}\t{fwd_doy}\t{fwd_date_str}\t{o3_val:.2f}\t{o3_date_str}\n")

        return exp_name, True
    except Exception as e:
        return exp_name, str(e)

def export_all_metrics():
    print("\n================ METRICS EXPORT =================\n")
    all_exp_names = sorted(set(exp_name for spec in FIGURE_SPECS for exp_name, _, _, _ in spec["experiments"]))

    for exp in tqdm(all_exp_names, desc="Exporting Metrics TXT"):
        exp_name, result = process_metrics_for_exp(exp)
        if result is not True:
            print(f"FAILED to export metrics for {exp_name}: {result}")
        gc.collect()

# ============================================================
# RUN: generate intermediate files only
# ============================================================

# 1) O3 cache
build_diag_cache_suite(
    varname="O3",
    diag_name="O3",
    compute_func=compute_o3_pc_ts,
    force_rebuild_flag=FORCE_REBUILD_O3,
)

# 2) Tmin50 cache
build_diag_cache_suite(
    varname="T",
    diag_name="Tmin50",
    compute_func=compute_polar_min_T50_ts,
    force_rebuild_flag=FORCE_REBUILD_T,
)

# 3) U60N10 cache
build_diag_cache_suite(
    varname="U",
    diag_name="U60N10",
    compute_func=compute_U60N10_ts,
    force_rebuild_flag=FORCE_REBUILD_U,
)

# 4) U60N50 cache (for metrics only; no plotting)
build_diag_cache_suite(
    varname="U",
    diag_name="U60N50",
    compute_func=compute_U60N50_ts,
    force_rebuild_flag=FORCE_REBUILD_U50,
)

# 5) metrics txt
export_all_metrics()

print("\nAll intermediate files generated successfully.")

In [ ]:
# ============================================================
# NOTEBOOK CELL 2
# Plot only:
#   - read from cache
#   - generate figures
# No recomputation of climatology / hindcast diagnostics.
# ============================================================

import os
import gc
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from tqdm import tqdm

# ------------------------------------------------------------
# Plot controls
# ------------------------------------------------------------
# 默认全画
ONLY_DIAGS = ["O3", "Tmin50", "U60N10"]

# 例如只想画某几个 figure，可写成：
# ONLY_PREFIXES = {"O3_evolution_fig13", "O3_evolution_fig14"}
ONLY_PREFIXES = None

# 如果想只画某一个 year，也可以改成比如 {"0008"}
ONLY_YEARS = None

# plot styling / y-limits
MEAN_LINEWIDTH = 5.0
MEMBER_LINEWIDTH_RATIO = 0.3
MEMBER_ALPHA = 0.3
SIGMA_ALPHA = 0.40

O3_YLIM = (60, 140)
TMIN50_YLIM = (175, 255)
U60N10_YLIM = (-40, 80)

# 全局路径和范围变量占位（请确保这些变量在你的实际环境中正确设置）
DATA_ROOT = "/mnt/soclim0/public_data/weiji"
OUT_ROOT = "/home/weiji/restart_exam/code/20260415egu/plots/hindcast/O3"
CACHE_ROOT = os.path.join(OUT_ROOT, "cache")
FIG_ROOT = os.path.join(OUT_ROOT, "figures")
XLIM_START = 0
XLIM_END = 150

# ------------------------------------------------------------
# Figure Specs 配置 (专门用于控制画图的组合和图例名称)
# ------------------------------------------------------------
FIGURE_SPECS = [
    # ---- 基础扰动对比 ----
    {
        "year": "0008", "prefix": "O3_evolution_fig1",
        "experiments": [
            ("0008-01", "Small Perturbation (Jan)", 0,  "forestgreen"),
            ("0008-02", "Small Perturbation (Feb)", 31, "royalblue"),
            ("0008-03", "Small Perturbation (Mar)", 59, "hotpink"),
        ],
    },
    {
        "year": "0008", "prefix": "O3_evolution_fig2",
        "experiments": [
            ("0008-02_v2", "Large Perturbation (Feb)", 31, "forestgreen"),
            ("0008-03_v2", "Large Perturbation (Mar)", 59, "royalblue"),
            ("0008-04_v2", "Large Perturbation (Apr)", 90, "hotpink"),
        ],
    },
    {
        "year": "0008", "prefix": "O3_evolution_fig3",
        "experiments": [
            ("0008-02",    "Small Perturbation", 31, "forestgreen"),
            ("0008-02_v2", "Large Perturbation", 31, "royalblue"),
            ("0008-02_v3", "Moisture Perturbation", 31, "hotpink"),
        ],
    },
    {
        "year": "0008", "prefix": "O3_evolution_fig4",
        "experiments": [
            ("0008-03",    "Small Perturbation", 59, "forestgreen"),
            ("0008-03_v2", "Large Perturbation", 59, "royalblue"),
            ("0008-03_v3", "Moisture Perturbation", 59, "hotpink"),
        ],
    },
    # ---- 其他年份的 Small Pert 对比 ----
    {
        "year": "0003", "prefix": "O3_evolution_fig5",
        "experiments": [
            ("0003-02", "Small Perturbation (Feb)", 31, "forestgreen"),
            ("0003-03", "Small Perturbation (Mar)", 59, "royalblue"),
        ],
    },
    {
        "year": "0013", "prefix": "O3_evolution_fig6",
        "experiments": [
            ("0013-02", "Small Perturbation (Feb)", 31, "forestgreen"),
            ("0013-03", "Small Perturbation (Mar)", 59, "royalblue"),
        ],
    },
    {
        "year": "0014", "prefix": "O3_evolution_fig7",
        "experiments": [
            ("0014-02", "Small Perturbation (Feb)", 31, "forestgreen"),
            ("0014-03", "Small Perturbation (Mar)", 59, "royalblue"),
        ],
    },
    {
        "year": "0019", "prefix": "O3_evolution_fig8",
        "experiments": [
            ("0019-02", "Small Perturbation (Feb)", 31, "forestgreen"),
            ("0019-03", "Small Perturbation (Mar)", 59, "royalblue"),
        ],
    },
    # ---- Nocouple 实验对比 ----
    {
        "year": "0008", "prefix": "O3_evolution_fig9",
        "experiments": [
            ("0008-02_NOCOUPL", "Prescribed O$_3$ (Feb)", 31, "forestgreen"),
            ("0008-03_NOCOUPL", "Prescribed O$_3$ (Mar)", 59, "royalblue"),
        ],
    },
    {
        "year": "0013", "prefix": "O3_evolution_fig10",
        "experiments": [
            ("0013-02_NOCOUPL", "Prescribed O$_3$ (Feb)", 31, "forestgreen"),
            ("0013-03_NOCOUPL", "Prescribed O$_3$ (Mar)", 59, "royalblue"),
        ],
    },
    {
        "year": "0014", "prefix": "O3_evolution_fig11",
        "experiments": [
            ("0014-02_NOCOUPL", "Prescribed O$_3$ (Feb)", 31, "forestgreen"),
            ("0014-03_NOCOUPL", "Prescribed O$_3$ (Mar)", 59, "royalblue"),
        ],
    },
    {
        "year": "0019", "prefix": "O3_evolution_fig12",
        "experiments": [
            ("0019-02_NOCOUPL", "Prescribed O$_3$ (Feb)", 31, "forestgreen"),
            ("0019-03_NOCOUPL", "Prescribed O$_3$ (Mar)", 59, "royalblue"),
        ],
    },
    # ---- 核心对比：Interactive vs Prescribed ----
    {
        "year": "0008", "prefix": "O3_evolution_fig13",
        "experiments": [
            ("0008-02",         "Interactive O$_3$", 31, "forestgreen"),
            ("0008-02_NOCOUPL", "Prescribed O$_3$",  31, "royalblue"),
        ],
    },
    {
        "year": "0008", "prefix": "O3_evolution_fig14",
        "experiments": [
            ("0008-03",         "Interactive O$_3$", 59, "forestgreen"),
            ("0008-03_NOCOUPL", "Prescribed O$_3$",  59, "royalblue"),
        ],
    },
    {
        "year": "0013", "prefix": "O3_evolution_fig15",
        "experiments": [
            ("0013-02",         "Interactive O$_3$", 31, "forestgreen"),
            ("0013-02_NOCOUPL", "Prescribed O$_3$",  31, "royalblue"),
        ],
    },
    {
        "year": "0013", "prefix": "O3_evolution_fig16",
        "experiments": [
            ("0013-03",         "Interactive O$_3$", 59, "forestgreen"),
            ("0013-03_NOCOUPL", "Prescribed O$_3$",  59, "royalblue"),
        ],
    },
    {
        "year": "0014", "prefix": "O3_evolution_fig17",
        "experiments": [
            ("0014-02",         "Interactive O$_3$", 31, "forestgreen"),
            ("0014-02_NOCOUPL", "Prescribed O$_3$",  31, "royalblue"),
        ],
    },
    {
        "year": "0014", "prefix": "O3_evolution_fig18",
        "experiments": [
            ("0014-03",         "Interactive O$_3$", 59, "forestgreen"),
            ("0014-03_NOCOUPL", "Prescribed O$_3$",  59, "royalblue"),
        ],
    },
    {
        "year": "0019", "prefix": "O3_evolution_fig19",
        "experiments": [
            ("0019-02",         "Interactive O$_3$", 31, "forestgreen"),
            ("0019-02_NOCOUPL", "Prescribed O$_3$",  31, "royalblue"),
        ],
    },
    {
        "year": "0019", "prefix": "O3_evolution_fig20",
        "experiments": [
            ("0019-03",         "Interactive O$_3$", 59, "forestgreen"),
            ("0019-03_NOCOUPL", "Prescribed O$_3$",  59, "royalblue"),
        ],
    },
]

# ------------------------------------------------------------
# Simple cache loader
# ------------------------------------------------------------
def open_cached_dataarray_local(path):
    da = xr.open_dataarray(path)
    da.load()
    return da

# ------------------------------------------------------------
# Plot function
# ------------------------------------------------------------
def plot_series_figure(ref_data, clim_data, experiments, year,
                       ylabel, ylim, out_subdir, output_filename_prefix,
                       xlim_start=0, xlim_end=150):
    all_months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                  "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    all_month_ticks = [0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334]

    xticks = [tick for tick in all_month_ticks if xlim_start <= tick <= xlim_end]
    xtick_labels = [all_months[i] for i, tick in enumerate(all_month_ticks) if tick in xticks]

    fig, ax = plt.subplots(figsize=(12, 8), constrained_layout=True)

    # --- 1. 绘制参考线 (Reference) 和 气候态 (Climatology) ---
    nref = ref_data.sizes["time"]
    ref_end = min(xlim_end, nref)
    ax.plot(
        np.arange(xlim_start, ref_end),
        ref_data.isel(time=slice(xlim_start, ref_end)).values,
        color="black", linewidth=MEAN_LINEWIDTH, label="Reference",
        zorder=10
    )

    clim_dim = list(clim_data.dims)[0]
    nclim = clim_data.sizes[clim_dim]
    clim_end = min(xlim_end, nclim)
    ax.plot(
        np.arange(xlim_start, clim_end),
        clim_data.isel({clim_dim: slice(xlim_start, clim_end)}).values,
        color="black", linestyle=":", linewidth=MEAN_LINEWIDTH, label="Climatology",
        zorder=9
    )

    # --- 2. 绘制实验组 ---
    member_lw = MEAN_LINEWIDTH * MEMBER_LINEWIDTH_RATIO

    for exp in experiments:
        data = exp["data"]
        label = exp["label"]  # 直接读取配置好的图例名称
        offset = exp["offset"]
        color = exp["color"]

        if offset >= xlim_end:
            continue

        total = data.sizes["time"]
        start_idx = max(0, xlim_start - offset)
        end_idx = min(total, xlim_end - offset)
        if end_idx <= start_idx:
            continue

        x = np.arange(offset + start_idx, offset + end_idx)
        sub = data.isel(time=slice(start_idx, end_idx))

        if "member" in sub.dims:
            plot_data = sub.transpose("member", "time")
            mean_line = plot_data.mean("member")
            std_line = plot_data.std("member")
            lower = mean_line - std_line
            upper = mean_line + std_line

            for i in range(plot_data.sizes["member"]):
                ax.plot(
                    x, plot_data.isel(member=i).values,
                    color=color, linewidth=member_lw, alpha=MEMBER_ALPHA,
                    zorder=2
                )

            ax.fill_between(x, lower.values, upper.values, color=color, alpha=SIGMA_ALPHA, zorder=1)            
            ax.plot(x, lower.values, color=color, linewidth=1, alpha=1, zorder=2)
            ax.plot(x, upper.values, color=color, linewidth=1, alpha=1, zorder=2)
            ax.plot(x, mean_line.values, color=color, linewidth=MEAN_LINEWIDTH, label=label, zorder=3)
        else:
            ax.plot(x, sub.values, color=color, linewidth=MEAN_LINEWIDTH, label=label, zorder=3)

    # --- 3. 细节微调 ---
    ax.set_xticks(xticks)
    ax.set_xticklabels(xtick_labels, fontsize=16)
    ax.set_xlim(xlim_start, xlim_end)
    ax.set_ylabel(ylabel, fontsize=18)
    ax.set_ylim(*ylim)
    ax.tick_params(axis="y", labelsize=16)

    ax.text(
        0.02, 0.95, f"Year: {year}",
        transform=ax.transAxes, fontsize=18, verticalalignment="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5)
    )

    ax.legend(fontsize=13)

    fig_dir = os.path.join(FIG_ROOT, out_subdir)
    os.makedirs(fig_dir, exist_ok=True)

    fig.savefig(os.path.join(fig_dir, f"{output_filename_prefix}_{year}.pdf"))
    fig.savefig(os.path.join(fig_dir, f"{output_filename_prefix}_{year}.png"), dpi=200)

    return fig, ax

# ------------------------------------------------------------
# Load cache + plot
# ------------------------------------------------------------
def plot_suite_from_cache(diag_name, ylabel, ylim, out_subdir):
    print(f"\n================ PLOT {diag_name} FROM CACHE =================\n")

    years_needed = sorted(set(spec["year"] for spec in FIGURE_SPECS))
    all_exp_names = sorted(set(exp_name for spec in FIGURE_SPECS for exp_name, _, _, _ in spec["experiments"]))

    clim_path = os.path.join(CACHE_ROOT, diag_name, f"{diag_name}_B2000WCN_climatology.nc")
    if not os.path.exists(clim_path):
        raise FileNotFoundError(f"Missing climatology cache: {clim_path}")

    clim = open_cached_dataarray_local(clim_path)

    ref_cache = {}
    for year in years_needed:
        ref_path = os.path.join(CACHE_ROOT, diag_name, f"{diag_name}_BWCN_{year}.nc")
        if not os.path.exists(ref_path):
            raise FileNotFoundError(f"Missing BWCN cache: {ref_path}")
        ref_cache[year] = open_cached_dataarray_local(ref_path)

    exp_cache = {}
    for exp_name in all_exp_names:
        exp_path = os.path.join(CACHE_ROOT, diag_name, f"{diag_name}_Hindcast_{exp_name}.nc")
        if not os.path.exists(exp_path):
            raise FileNotFoundError(f"Missing Hindcast cache: {exp_path}")
        exp_cache[exp_name] = open_cached_dataarray_local(exp_path)

    for spec in tqdm(FIGURE_SPECS, desc=f"Plotting {diag_name}"):
        if ONLY_YEARS is not None and spec["year"] not in ONLY_YEARS:
            continue

        prefix = spec["prefix"]
        if ONLY_PREFIXES is not None and prefix not in ONLY_PREFIXES:
            continue

        plot_prefix = prefix
        if diag_name != "O3":
            plot_prefix = plot_prefix.replace("O3_evolution", f"{diag_name}_evolution")

        experiments = []
        for exp_name, label, offset, color in spec["experiments"]:
            experiments.append({
                "data": exp_cache[exp_name],
                "label": label,
                "offset": offset,
                "color": color,
            })

        fig, ax = plot_series_figure(
            ref_data=ref_cache[spec["year"]],
            clim_data=clim,
            experiments=experiments,
            year=spec["year"],
            ylabel=ylabel,
            ylim=ylim,
            out_subdir=out_subdir,
            output_filename_prefix=plot_prefix,
            xlim_start=XLIM_START,
            xlim_end=XLIM_END,
        )
        plt.close(fig)
        gc.collect()

# ------------------------------------------------------------
# Run plotting
# ------------------------------------------------------------
if "O3" in ONLY_DIAGS:
    plot_suite_from_cache(
        diag_name="O3",
        ylabel="Partial ozone column, 30–70 hPa (DU)",
        ylim=O3_YLIM,
        out_subdir="O3",
    )

if "Tmin50" in ONLY_DIAGS:
    plot_suite_from_cache(
        diag_name="Tmin50",
        ylabel="Polar minimum T at 50 hPa (K)",
        ylim=TMIN50_YLIM,
        out_subdir="Tmin50",
    )

if "U60N10" in ONLY_DIAGS:
    plot_suite_from_cache(
        diag_name="U60N10",
        ylabel="Zonal-mean U at 60°N, 10 hPa (m s$^{-1}$)",
        ylim=U60N10_YLIM,
        out_subdir="U60N10",
    )

print("\nAll figures generated from cache.")

In [ ]:
# ============================================================
# NOTEBOOK CELL: Plot one scatter per experiment from metrics txt
# X = absolute FWD_DOY (corrected from relative lead days in metrics.txt)
# Y = O3_MA_Min_Val
# Same year => same xlim / ylim
# No low25 comparison
# Only one mean FWD line + one ±1σ band
# + BWCN reference star
# ============================================================

import os
import re
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.stats import pearsonr

# ============================================================
# Paths & Controls
# ============================================================

HINDCAST_ROOT = "/mnt/soclim0/public_data/weiji/Hindcast"
BWCN_FWD_TXT = "/mnt/soclim0/public_data/weiji/BWCN/BWCN_final_warming_date.txt"

OUT_ROOT = "/home/weiji/restart_exam/code/20260415egu/plots/hindcast/O3"
CACHE_ROOT = os.path.join(OUT_ROOT, "cache")
OUT_DIR = os.path.join(OUT_ROOT, "figures", "SCATTER_FWD_vs_O3MIN")
os.makedirs(OUT_DIR, exist_ok=True)

# 只画某些年份；None = 全部
ONLY_YEARS = None
# 例如 {"0008", "0013"}

# 只画某些实验；None = 全部
ONLY_EXPS = None
# 例如 {"0008-02", "0008-03", "0008-02_NOCOUPL"}

SHOW_FIG = False

USE_MANUAL_AXIS_LIMITS = False

# 如果手动指定，可按 year 单独给
MANUAL_XLIM_BY_YEAR = {
    # "0008": (60, 150),
}
MANUAL_YLIM_BY_YEAR = {
    # "0008": (60, 170),
}

X_PAD = 4.0
Y_PAD = 3.0

FIG_DPI = 220
FIGSIZE = (6.4, 5.3)

APPLY_O3_5D = True

# ============================================================
# Style
# ============================================================

POINT_COLOR = "0.35"
POINT_ALPHA = 0.80
POINT_SIZE = 36

MEAN_COLOR = "#1f77b4"
BAND_COLOR = "#1f77b4"
MEAN_LINEWIDTH = 1.5
BAND_ALPHA = 0.14

REF_STAR_COLOR = "black"
REF_STAR_SIZE = 170
REF_STAR_EDGEWIDTH = 0.8

GRID_ALPHA = 0.45
TITLE_FONTSIZE = 13
LABEL_FONTSIZE = 12
TICK_FONTSIZE = 10
TEXT_FONTSIZE = 9.5
LEGEND_FONTSIZE = 8.2

# ============================================================
# Helpers
# ============================================================

def doy_to_mmdd(x, pos):
    if np.isnan(x) or x < 1 or x > 365:
        return ""
    base_date = datetime(2001, 1, 1)
    target_date = base_date + timedelta(days=int(round(x)) - 1)
    return target_date.strftime("%m-%d")

def discover_experiments_with_metrics(hindcast_root):
    exps = []
    for d in sorted(os.listdir(hindcast_root)):
        full = os.path.join(hindcast_root, d)
        if not os.path.isdir(full):
            continue
        txt = os.path.join(full, f"{d}_metrics.txt")
        if os.path.exists(txt):
            exps.append(d)
    return exps

def load_fwd_txt(txt_path):
    fwd_dict = {}
    if not os.path.exists(txt_path):
        print(f"Warning: missing FWD txt -> {txt_path}")
        return fwd_dict

    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if (not line) or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) >= 2:
                year = int(parts[0])
                doy_str = parts[1]
                if doy_str.lower() != "nan":
                    fwd_dict[year] = float(doy_str)
    return fwd_dict

BWCN_FWD_DICT = load_fwd_txt(BWCN_FWD_TXT)

def get_init_offset_from_exp_name(exp_name):
    m = re.match(r"^\d{4}-(\d{2})", exp_name)
    if m is None:
        raise ValueError(f"Cannot infer init month from exp_name: {exp_name}")

    mm = int(m.group(1))
    month_start_doy = {
        1: 0,
        2: 31,
        3: 59,
        4: 90,
    }
    if mm not in month_start_doy:
        raise ValueError(f"Unsupported init month in exp_name: {exp_name}")
    return month_start_doy[mm]

def load_metrics_txt(metrics_txt, exp_name):
    """
    读取 metrics txt，并把相对 FWD_DOY 修正成全年绝对 DOY
    """
    if not os.path.exists(metrics_txt):
        print(f"Warning: missing metrics txt -> {metrics_txt}")
        return None

    df = pd.read_csv(metrics_txt, sep=r"\s+", engine="python")

    required = ["Member", "FWD_DOY", "O3_MA_Min_Val"]
    for c in required:
        if c not in df.columns:
            raise ValueError(f"{metrics_txt} missing column: {c}")

    df["Member"] = pd.to_numeric(df["Member"], errors="coerce")
    df["FWD_DOY"] = pd.to_numeric(df["FWD_DOY"], errors="coerce")
    df["O3_MA_Min_Val"] = pd.to_numeric(df["O3_MA_Min_Val"], errors="coerce")

    df = df[np.isfinite(df["FWD_DOY"]) & np.isfinite(df["O3_MA_Min_Val"])].copy()
    df = df.sort_values("Member").reset_index(drop=True)

    df["FWD_DOY_REL"] = df["FWD_DOY"].astype(float)
    df["FWD_DOY_ABS"] = df["FWD_DOY_REL"] + get_init_offset_from_exp_name(exp_name)

    return df

def build_experiment_df(exp_name):
    metrics_txt = os.path.join(HINDCAST_ROOT, exp_name, f"{exp_name}_metrics.txt")
    df = load_metrics_txt(metrics_txt, exp_name)
    if df is None or len(df) == 0:
        return None

    df["exp_name"] = exp_name
    df["year"] = exp_name[:4]
    return df

_BWCN_REF_O3MIN_CACHE = {}

def get_bwcn_reference_o3min_for_year(year_str, apply_o3_5d=True):
    cache_key = (year_str, bool(apply_o3_5d))
    if cache_key in _BWCN_REF_O3MIN_CACHE:
        return _BWCN_REF_O3MIN_CACHE[cache_key]

    o3_nc = os.path.join(CACHE_ROOT, "O3", f"O3_BWCN_{year_str}.nc")
    if not os.path.exists(o3_nc):
        print(f"Warning: missing BWCN O3 cache -> {o3_nc}")
        _BWCN_REF_O3MIN_CACHE[cache_key] = np.nan
        return np.nan

    da = xr.open_dataarray(o3_nc).load().sortby("time")

    if apply_o3_5d:
        da = da.rolling(time=5, center=True, min_periods=5).mean()

    mask = da.time.dt.month.isin([3, 4])
    seg = da.where(mask, drop=True)

    if seg.size == 0 or int(seg.count().values) == 0:
        val = np.nan
    else:
        val = float(seg.min().values)

    da.close()
    _BWCN_REF_O3MIN_CACHE[cache_key] = val
    return val

def get_reference_star_for_year(year_str):
    year_int = int(year_str)
    x_ref = BWCN_FWD_DICT.get(year_int, np.nan)
    y_ref = get_bwcn_reference_o3min_for_year(year_str, apply_o3_5d=APPLY_O3_5D)
    return x_ref, y_ref

def compute_year_axis_limits(exp_dfs_by_year):
    """
    同一年所有实验共用同一套 x/y limit，并把 BWCN 参考星号也纳入范围
    """
    limits = {}

    for year, exp_map in exp_dfs_by_year.items():
        if USE_MANUAL_AXIS_LIMITS and year in MANUAL_XLIM_BY_YEAR and year in MANUAL_YLIM_BY_YEAR:
            limits[year] = {
                "xlim": MANUAL_XLIM_BY_YEAR[year],
                "ylim": MANUAL_YLIM_BY_YEAR[year],
            }
            continue

        all_x = []
        all_y = []

        for exp_name, df in exp_map.items():
            if df is None or len(df) == 0:
                continue
            all_x.extend(df["FWD_DOY_ABS"].values.astype(float))
            all_y.extend(df["O3_MA_Min_Val"].values.astype(float))

        x_ref, y_ref = get_reference_star_for_year(year)
        if np.isfinite(x_ref) and np.isfinite(y_ref):
            all_x.append(float(x_ref))
            all_y.append(float(y_ref))

        all_x = np.asarray(all_x, dtype=float)
        all_y = np.asarray(all_y, dtype=float)

        if all_x.size == 0 or all_y.size == 0:
            limits[year] = {"xlim": (60, 150), "ylim": (60, 170)}
            continue

        xmin = np.nanmin(all_x) - X_PAD
        xmax = np.nanmax(all_x) + X_PAD
        ymin = np.nanmin(all_y) - Y_PAD
        ymax = np.nanmax(all_y) + Y_PAD

        limits[year] = {
            "xlim": (xmin, xmax),
            "ylim": (ymin, ymax),
        }

    return limits

# ============================================================
# Plotting
# ============================================================

def draw_one_experiment_scatter(ax, df, exp_name, xlim, ylim, ref_xy=None, ref_year=None):
    ax.grid(True, linestyle=":", alpha=GRID_ALPHA)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    if df is None or len(df) == 0:
        ax.set_title(exp_name, fontsize=TITLE_FONTSIZE, fontweight="bold")
        ax.text(0.5, 0.5, "No valid data", ha="center", va="center", transform=ax.transAxes)
        ax.set_xlabel("Final Warming Date", fontsize=LABEL_FONTSIZE)
        ax.set_ylabel(r"MA min O$_3$ (DU)", fontsize=LABEL_FONTSIZE)
        ax.xaxis.set_major_formatter(FuncFormatter(doy_to_mmdd))
        return

    x = df["FWD_DOY_ABS"].values.astype(float)
    y = df["O3_MA_Min_Val"].values.astype(float)

    # scatter
    ax.scatter(
        x, y,
        s=POINT_SIZE,
        color=POINT_COLOR,
        alpha=POINT_ALPHA,
        edgecolors="none",
        zorder=3
    )

    # mean FWD line + ±1σ band
    mean_fwd = np.nanmean(x)
    std_fwd = np.nanstd(x, ddof=0)

    ax.axvspan(
        mean_fwd - std_fwd,
        mean_fwd + std_fwd,
        color=BAND_COLOR,
        alpha=BAND_ALPHA,
        zorder=0
    )
    ax.axvline(
        mean_fwd,
        color=MEAN_COLOR,
        linestyle="-",
        linewidth=MEAN_LINEWIDTH,
        alpha=0.95,
        zorder=1
    )

    # BWCN reference star
    x_ref, y_ref = (np.nan, np.nan) if ref_xy is None else ref_xy
    if np.isfinite(x_ref) and np.isfinite(y_ref):
        ax.scatter(
            x_ref, y_ref,
            s=REF_STAR_SIZE,
            marker="*",
            color=REF_STAR_COLOR,
            edgecolors="white",
            linewidths=REF_STAR_EDGEWIDTH,
            zorder=6
        )

    # statistics
    n = len(x)
    if n >= 3:
        try:
            r, p = pearsonr(x, y)
            txt = f"r = {r:.3f}\np = {p:.2e}\nN = {n}"
        except Exception:
            txt = f"N = {n}"
    else:
        txt = f"N = {n}"

    ax.text(
        0.04, 0.96, txt,
        transform=ax.transAxes,
        va="top", ha="left",
        fontsize=TEXT_FONTSIZE,
        color="k",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.40, edgecolor="0.7")
    )

    # legend
    handles = [
        Line2D([0], [0], marker="o", color="none",
               markerfacecolor=POINT_COLOR, markeredgecolor="none",
               markersize=6.5, alpha=POINT_ALPHA),
        Line2D([0], [0], color=MEAN_COLOR, lw=MEAN_LINEWIDTH, ls="-"),
        Patch(facecolor=BAND_COLOR, edgecolor="none", alpha=BAND_ALPHA),
    ]
    labels = [
        "Members",
        f"Mean FWD: {doy_to_mmdd(mean_fwd, None)}",
        r"FWD $\pm 1\sigma$",
    ]

    if np.isfinite(x_ref) and np.isfinite(y_ref):
        handles.append(
            Line2D([0], [0], marker="*", color="none",
                   markerfacecolor=REF_STAR_COLOR, markeredgecolor="white",
                   markeredgewidth=0.8, markersize=11, linewidth=0)
        )
        labels.append(f"BWCN reference ({ref_year})")

    ax.legend(
        handles, labels,
        loc="lower left",
        fontsize=LEGEND_FONTSIZE,
        frameon=False,
        handletextpad=0.45,
        borderpad=0.2,
        labelspacing=0.3
    )

    ax.set_title(exp_name, fontsize=TITLE_FONTSIZE, fontweight="bold")
    ax.set_xlabel("Final Warming Date", fontsize=LABEL_FONTSIZE)
    ax.set_ylabel(r"MA min O$_3$ (DU)", fontsize=LABEL_FONTSIZE)
    ax.xaxis.set_major_formatter(FuncFormatter(doy_to_mmdd))
    ax.tick_params(axis="both", labelsize=TICK_FONTSIZE)

    for tick in ax.get_xticklabels():
        tick.set_rotation(45)

def save_one_experiment_plot(df, exp_name, xlim, ylim):
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=FIG_DPI)

    year = exp_name[:4]
    ref_xy = get_reference_star_for_year(year)

    draw_one_experiment_scatter(
        ax, df, exp_name, xlim, ylim,
        ref_xy=ref_xy,
        ref_year=year
    )

    fig.suptitle(
        f"O$_3$ minimum vs Final Warming Date ({year})",
        fontsize=14, fontweight="bold", y=1.02
    )

    png_path = os.path.join(OUT_DIR, f"Scatter_FWD_vs_O3MIN_{exp_name}.png")
    pdf_path = os.path.join(OUT_DIR, f"Scatter_FWD_vs_O3MIN_{exp_name}.pdf")

    plt.savefig(png_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")

    if SHOW_FIG:
        plt.show()
    else:
        plt.close(fig)

    return png_path, pdf_path

# ============================================================
# Main
# ============================================================

print("Discovering experiments...")
all_exps = discover_experiments_with_metrics(HINDCAST_ROOT)

if ONLY_YEARS is not None:
    all_exps = [e for e in all_exps if e[:4] in ONLY_YEARS]

if ONLY_EXPS is not None:
    all_exps = [e for e in all_exps if e in ONLY_EXPS]

all_exps = sorted(all_exps)

print(f"Found {len(all_exps)} experiments with metrics txt.")
if len(all_exps) == 0:
    raise RuntimeError("No experiment metrics txt found.")

# load all dfs
exp_dfs = {}
for exp_name in all_exps:
    exp_dfs[exp_name] = build_experiment_df(exp_name)

# group by year
exp_dfs_by_year = {}
for exp_name, df in exp_dfs.items():
    year = exp_name[:4]
    exp_dfs_by_year.setdefault(year, {})
    exp_dfs_by_year[year][exp_name] = df

# compute per-year shared limits
year_limits = compute_year_axis_limits(exp_dfs_by_year)

print("\nPer-year shared limits:")
for year in sorted(year_limits):
    print(year, "xlim =", year_limits[year]["xlim"], "ylim =", year_limits[year]["ylim"])

# plot & save
print("\nPlotting...")
saved_files = []

for year in sorted(exp_dfs_by_year):
    xlim = year_limits[year]["xlim"]
    ylim = year_limits[year]["ylim"]

    for exp_name in sorted(exp_dfs_by_year[year]):
        df = exp_dfs_by_year[year][exp_name]
        png_path, pdf_path = save_one_experiment_plot(df, exp_name, xlim, ylim)
        saved_files.append((png_path, pdf_path))

print(f"\nDone. Saved {len(saved_files)} experiment scatter plots to:\n{OUT_DIR}")

In [ ]:
# ============================================================
# NOTEBOOK CELL: Pair scatter plots
# small_pert (coupled) vs NOCOUPL
# + BWCN reference star
# X = absolute FWD_DOY (corrected from relative lead days in metrics.txt)
# Y = O3_MA_Min_Val
# Each pair in one figure
# ============================================================

import os
import re
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.stats import pearsonr

# ============================================================
# Paths
# ============================================================

HINDCAST_ROOT = "/mnt/soclim0/public_data/weiji/Hindcast"
BWCN_FWD_TXT = "/mnt/soclim0/public_data/weiji/BWCN/BWCN_final_warming_date.txt"

OUT_ROOT = "/home/weiji/restart_exam/code/20260415egu/plots/hindcast/O3"
CACHE_ROOT = os.path.join(OUT_ROOT, "cache")
OUT_DIR = os.path.join(OUT_ROOT, "figures", "SCATTER_pair_small_vs_nocouple")
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# Which pairs to plot
# ============================================================

PAIR_SPECS = [
    ("0008-02", "0008-02_NOCOUPL"),
    ("0008-03", "0008-03_NOCOUPL"),
    ("0013-02", "0013-02_NOCOUPL"),
    ("0013-03", "0013-03_NOCOUPL"),
    ("0014-02", "0014-02_NOCOUPL"),
    ("0014-03", "0014-03_NOCOUPL"),
    ("0019-02", "0019-02_NOCOUPL"),
    ("0019-03", "0019-03_NOCOUPL"),
]

# 只调某几组时可改成子集；None 表示全部
ONLY_PAIRS = None
# 示例：
# ONLY_PAIRS = {
#     ("0008-02", "0008-02_NOCOUPL"),
#     ("0008-03", "0008-03_NOCOUPL"),
# }

SHOW_FIG = False
FIG_DPI = 220
FIGSIZE = (7.0, 5.8)

USE_MANUAL_AXIS_LIMITS = False
MANUAL_XLIM = (60, 150)
MANUAL_YLIM = (60, 170)

X_PAD = 4.0
Y_PAD = 3.0

APPLY_O3_5D = True

# ============================================================
# Style
# ============================================================

COUPLED_COLOR = "#d62728"   # red
NOCOUPL_COLOR = "#1f77b4"   # blue

COUPLED_ALPHA = 0.85
NOCOUPL_ALPHA = 0.85
POINT_SIZE = 42

COUPLED_BAND_ALPHA = 0.14
NOCOUPL_BAND_ALPHA = 0.12
MEAN_LINEWIDTH = 1.5

REF_STAR_COLOR = "black"
REF_STAR_SIZE = 180
REF_STAR_EDGEWIDTH = 0.8

GRID_ALPHA = 0.45
TITLE_FONTSIZE = 13
LABEL_FONTSIZE = 12
TICK_FONTSIZE = 10
TEXT_FONTSIZE = 9.2
LEGEND_FONTSIZE = 8.0

# ============================================================
# Helpers
# ============================================================

def doy_to_mmdd(x, pos):
    if np.isnan(x) or x < 1 or x > 365:
        return ""
    base_date = datetime(2001, 1, 1)
    target_date = base_date + timedelta(days=int(round(x)) - 1)
    return target_date.strftime("%m-%d")

def load_fwd_txt(txt_path):
    fwd_dict = {}
    if not os.path.exists(txt_path):
        print(f"Warning: missing FWD txt -> {txt_path}")
        return fwd_dict

    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if (not line) or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) >= 2:
                year = int(parts[0])
                doy_str = parts[1]
                if doy_str.lower() != "nan":
                    fwd_dict[year] = float(doy_str)
    return fwd_dict

BWCN_FWD_DICT = load_fwd_txt(BWCN_FWD_TXT)

def get_init_offset_from_exp_name(exp_name):
    """
    metrics.txt 里的 FWD_DOY 是“距离开始几天”，这里转成全年绝对 DOY
    规则：
      -01 -> +0
      -02 -> +31
      -03 -> +59
      -04 -> +90
    """
    m = re.match(r"^\d{4}-(\d{2})", exp_name)
    if m is None:
        raise ValueError(f"Cannot infer init month from exp_name: {exp_name}")

    mm = int(m.group(1))
    month_start_doy = {
        1: 0,
        2: 31,
        3: 59,
        4: 90,
    }
    if mm not in month_start_doy:
        raise ValueError(f"Unsupported init month in exp_name: {exp_name}")
    return month_start_doy[mm]

def load_metrics_txt(exp_name):
    metrics_txt = os.path.join(HINDCAST_ROOT, exp_name, f"{exp_name}_metrics.txt")
    if not os.path.exists(metrics_txt):
        print(f"Warning: missing metrics txt -> {metrics_txt}")
        return None

    df = pd.read_csv(metrics_txt, sep=r"\s+", engine="python")

    required = ["Member", "FWD_DOY", "O3_MA_Min_Val"]
    for c in required:
        if c not in df.columns:
            raise ValueError(f"{metrics_txt} missing column: {c}")

    df["Member"] = pd.to_numeric(df["Member"], errors="coerce")
    df["FWD_DOY"] = pd.to_numeric(df["FWD_DOY"], errors="coerce")
    df["O3_MA_Min_Val"] = pd.to_numeric(df["O3_MA_Min_Val"], errors="coerce")

    df = df[np.isfinite(df["FWD_DOY"]) & np.isfinite(df["O3_MA_Min_Val"])].copy()
    df = df.sort_values("Member").reset_index(drop=True)

    # 原始是相对起报日
    df["FWD_DOY_REL"] = df["FWD_DOY"].astype(float)

    # 修正成全年绝对 DOY
    offset = get_init_offset_from_exp_name(exp_name)
    df["FWD_DOY_ABS"] = df["FWD_DOY_REL"] + offset

    return df

_BWCN_REF_O3MIN_CACHE = {}

def get_bwcn_reference_o3min_for_year(year_str, apply_o3_5d=True):
    """
    从 BWCN O3 cache 里计算参考年的 3–4 月 O3 minimum
    """
    cache_key = (year_str, bool(apply_o3_5d))
    if cache_key in _BWCN_REF_O3MIN_CACHE:
        return _BWCN_REF_O3MIN_CACHE[cache_key]

    o3_nc = os.path.join(CACHE_ROOT, "O3", f"O3_BWCN_{year_str}.nc")
    if not os.path.exists(o3_nc):
        print(f"Warning: missing BWCN O3 cache -> {o3_nc}")
        _BWCN_REF_O3MIN_CACHE[cache_key] = np.nan
        return np.nan

    da = xr.open_dataarray(o3_nc).load().sortby("time")

    if apply_o3_5d:
        da = da.rolling(time=5, center=True, min_periods=5).mean()

    mask = da.time.dt.month.isin([3, 4])
    seg = da.where(mask, drop=True)

    if seg.size == 0 or int(seg.count().values) == 0:
        val = np.nan
    else:
        val = float(seg.min().values)

    da.close()
    _BWCN_REF_O3MIN_CACHE[cache_key] = val
    return val

def get_reference_star_for_year(year_str):
    """
    BWCN 参考星号位置:
      x_ref = BWCN reference FWD
      y_ref = BWCN reference O3 minimum
    """
    year_int = int(year_str)
    x_ref = BWCN_FWD_DICT.get(year_int, np.nan)
    y_ref = get_bwcn_reference_o3min_for_year(year_str, apply_o3_5d=APPLY_O3_5D)
    return x_ref, y_ref

def compute_pair_axis_limits(df1, df2, ref_xy=None):
    if USE_MANUAL_AXIS_LIMITS:
        return MANUAL_XLIM, MANUAL_YLIM

    all_x = []
    all_y = []

    for df in [df1, df2]:
        if df is not None and len(df) > 0:
            all_x.extend(df["FWD_DOY_ABS"].values.astype(float))
            all_y.extend(df["O3_MA_Min_Val"].values.astype(float))

    if ref_xy is not None:
        x_ref, y_ref = ref_xy
        if np.isfinite(x_ref) and np.isfinite(y_ref):
            all_x.append(float(x_ref))
            all_y.append(float(y_ref))

    all_x = np.asarray(all_x, dtype=float)
    all_y = np.asarray(all_y, dtype=float)

    if all_x.size == 0 or all_y.size == 0:
        return (60, 150), (60, 170)

    xmin = np.nanmin(all_x) - X_PAD
    xmax = np.nanmax(all_x) + X_PAD
    ymin = np.nanmin(all_y) - Y_PAD
    ymax = np.nanmax(all_y) + Y_PAD

    return (xmin, xmax), (ymin, ymax)

def stats_text(df):
    if df is None or len(df) == 0:
        return "N = 0"

    x = df["FWD_DOY_ABS"].values.astype(float)
    y = df["O3_MA_Min_Val"].values.astype(float)
    n = len(x)

    if n >= 3:
        try:
            r, p = pearsonr(x, y)
            return f"r = {r:.3f}\np = {p:.2e}\nN = {n}"
        except Exception:
            return f"N = {n}"
    return f"N = {n}"

# ============================================================
# Plotting
# ============================================================

def draw_pair_scatter(ax, df_cpl, df_noc, exp_cpl, exp_noc, xlim, ylim, ref_xy=None, ref_year=None):
    ax.grid(True, linestyle=":", alpha=GRID_ALPHA)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    # coupled
    if df_cpl is not None and len(df_cpl) > 0:
        x_c = df_cpl["FWD_DOY_ABS"].values.astype(float)
        y_c = df_cpl["O3_MA_Min_Val"].values.astype(float)

        ax.scatter(
            x_c, y_c,
            s=POINT_SIZE,
            color=COUPLED_COLOR,
            alpha=COUPLED_ALPHA,
            edgecolors="none",
            zorder=3
        )

        mean_c = np.nanmean(x_c)
        std_c = np.nanstd(x_c, ddof=0)

        ax.axvspan(
            mean_c - std_c, mean_c + std_c,
            color=COUPLED_COLOR,
            alpha=COUPLED_BAND_ALPHA,
            zorder=0
        )
        ax.axvline(
            mean_c,
            color=COUPLED_COLOR,
            linestyle="-",
            linewidth=MEAN_LINEWIDTH,
            alpha=0.95,
            zorder=1
        )
    else:
        mean_c = np.nan

    # nocouple
    if df_noc is not None and len(df_noc) > 0:
        x_n = df_noc["FWD_DOY_ABS"].values.astype(float)
        y_n = df_noc["O3_MA_Min_Val"].values.astype(float)

        ax.scatter(
            x_n, y_n,
            s=POINT_SIZE,
            color=NOCOUPL_COLOR,
            alpha=NOCOUPL_ALPHA,
            edgecolors="none",
            zorder=3
        )

        mean_n = np.nanmean(x_n)
        std_n = np.nanstd(x_n, ddof=0)

        ax.axvspan(
            mean_n - std_n, mean_n + std_n,
            color=NOCOUPL_COLOR,
            alpha=NOCOUPL_BAND_ALPHA,
            zorder=0
        )
        ax.axvline(
            mean_n,
            color=NOCOUPL_COLOR,
            linestyle="-",
            linewidth=MEAN_LINEWIDTH,
            alpha=0.95,
            zorder=1
        )
    else:
        mean_n = np.nan

    # BWCN reference star
    x_ref, y_ref = (np.nan, np.nan) if ref_xy is None else ref_xy
    if np.isfinite(x_ref) and np.isfinite(y_ref):
        ax.scatter(
            x_ref, y_ref,
            s=REF_STAR_SIZE,
            marker="*",
            color=REF_STAR_COLOR,
            edgecolors="white",
            linewidths=REF_STAR_EDGEWIDTH,
            zorder=6
        )

    # text boxes
    txt_c = stats_text(df_cpl)
    txt_n = stats_text(df_noc)

    ax.text(
        0.03, 0.96, "Coupled\n" + txt_c,
        transform=ax.transAxes,
        va="top", ha="left",
        fontsize=TEXT_FONTSIZE,
        color=COUPLED_COLOR,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.42, edgecolor=COUPLED_COLOR)
    )

    ax.text(
        0.97, 0.96, "NOCOUPL\n" + txt_n,
        transform=ax.transAxes,
        va="top", ha="right",
        fontsize=TEXT_FONTSIZE,
        color=NOCOUPL_COLOR,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.42, edgecolor=NOCOUPL_COLOR)
    )

    # legend
    handles = [
        Line2D([0], [0], marker="o", color="none",
               markerfacecolor=COUPLED_COLOR, markeredgecolor="none",
               markersize=6.5, alpha=COUPLED_ALPHA),
        Line2D([0], [0], marker="o", color="none",
               markerfacecolor=NOCOUPL_COLOR, markeredgecolor="none",
               markersize=6.5, alpha=NOCOUPL_ALPHA),
        Line2D([0], [0], color=COUPLED_COLOR, lw=MEAN_LINEWIDTH, ls="-"),
        Patch(facecolor=COUPLED_COLOR, edgecolor="none", alpha=COUPLED_BAND_ALPHA),
        Line2D([0], [0], color=NOCOUPL_COLOR, lw=MEAN_LINEWIDTH, ls="-"),
        Patch(facecolor=NOCOUPL_COLOR, edgecolor="none", alpha=NOCOUPL_BAND_ALPHA),
    ]
    labels = [
        f"Coupled ({exp_cpl})",
        f"NOCOUPL ({exp_noc})",
        f"Coupled mean FWD: {doy_to_mmdd(mean_c, None) if np.isfinite(mean_c) else 'NA'}",
        r"Coupled FWD $\pm 1\sigma$",
        f"NOCOUPL mean FWD: {doy_to_mmdd(mean_n, None) if np.isfinite(mean_n) else 'NA'}",
        r"NOCOUPL FWD $\pm 1\sigma$",
    ]

    if np.isfinite(x_ref) and np.isfinite(y_ref):
        handles.append(
            Line2D([0], [0], marker="*", color="none",
                   markerfacecolor=REF_STAR_COLOR, markeredgecolor="white",
                   markeredgewidth=0.8, markersize=11, linewidth=0)
        )
        labels.append(f"BWCN reference ({ref_year})")

    ax.legend(
        handles, labels,
        loc="lower left",
        fontsize=LEGEND_FONTSIZE,
        frameon=False,
        handletextpad=0.45,
        borderpad=0.2,
        labelspacing=0.3
    )

    ax.set_xlabel("Final Warming Date", fontsize=LABEL_FONTSIZE)
    ax.set_ylabel(r"MA min O$_3$ (DU)", fontsize=LABEL_FONTSIZE)
    ax.xaxis.set_major_formatter(FuncFormatter(doy_to_mmdd))
    ax.tick_params(axis="both", labelsize=TICK_FONTSIZE)

    for tick in ax.get_xticklabels():
        tick.set_rotation(45)

def save_pair_plot(exp_cpl, exp_noc):
    df_cpl = load_metrics_txt(exp_cpl)
    df_noc = load_metrics_txt(exp_noc)

    year = exp_cpl[:4]
    ref_xy = get_reference_star_for_year(year)

    xlim, ylim = compute_pair_axis_limits(df_cpl, df_noc, ref_xy=ref_xy)

    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=FIG_DPI)
    draw_pair_scatter(
        ax, df_cpl, df_noc, exp_cpl, exp_noc,
        xlim, ylim,
        ref_xy=ref_xy,
        ref_year=year
    )

    month_tag = exp_cpl[5:7]
    fig.suptitle(
        f"O$_3$ minimum vs Final Warming Date: {year}-{month_tag}  Coupled vs NOCOUPL",
        fontsize=14, fontweight="bold", y=1.02
    )

    base = f"Scatter_pair_FWD_vs_O3MIN_{exp_cpl}_vs_{exp_noc}"
    png_path = os.path.join(OUT_DIR, base + ".png")
    pdf_path = os.path.join(OUT_DIR, base + ".pdf")

    plt.savefig(png_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")

    if SHOW_FIG:
        plt.show()
    else:
        plt.close(fig)

    return png_path, pdf_path, xlim, ylim, ref_xy

# ============================================================
# Main
# ============================================================

pairs_to_plot = PAIR_SPECS
if ONLY_PAIRS is not None:
    pairs_to_plot = [p for p in PAIR_SPECS if p in ONLY_PAIRS]

print(f"Plotting {len(pairs_to_plot)} pair figures...")

saved = []
for exp_cpl, exp_noc in pairs_to_plot:
    png_path, pdf_path, xlim, ylim, ref_xy = save_pair_plot(exp_cpl, exp_noc)
    saved.append((exp_cpl, exp_noc, png_path, pdf_path, xlim, ylim, ref_xy))
    print(f"{exp_cpl} vs {exp_noc}")
    print("  xlim =", xlim)
    print("  ylim =", ylim)
    print("  BWCN ref star =", ref_xy)

print(f"\nDone. Saved {len(saved)} figures to:\n{OUT_DIR}")